In [ ]:
from google.colab import drive
import sys
import os

# Mount the shared drive
drive.mount('/content/drive')

# Define the project root
PROJECT_ROOT = '/content/drive/MyDrive/MultiCamera_Vehicle_ReIdentification'

# Change the working directory to the project root
os.chdir(PROJECT_ROOT)

# Add the 'src' folder to Python's path so you can import your custom modules
sys.path.append(os.path.join(PROJECT_ROOT, 'src'))

print(f"Current Working Directory: {os.getcwd()}")

In [ ]:
def preprocess_plate(plate_crop):
    """
    Cleans up the cropped license plate image to improve OCR accuracy.
    """
    # 1. Convert to grayscale
    gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)

    # 2. Resize to make the text larger for the OCR engine (scaling up by 2x)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    # 3. Apply bilateral filter to remove noise while keeping edges sharp
    blur = cv2.bilateralFilter(gray, 11, 17, 17)

    # 4. Optional: Adaptive thresholding to make text stand out against the background
    # (Comment this out if your specific dataset plates get too distorted)
    # processed = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

    return blur

In [ ]:
def extract_license_plate_text(image, bbox=None):
    """
    Takes a full image and a YOLO bounding box, crops it, and reads the text.
    If bbox is None, it assumes the image is ALREADY a cropped plate.
    """
    # 1. Crop the image if a bounding box is provided
    if bbox is not None:
        x_min, y_min, x_max, y_max = [int(val) for val in bbox]
        # Ensure coordinates are within image boundaries
        x_min, y_min = max(0, x_min), max(0, y_min)
        plate_crop = image[y_min:y_max, x_min:x_max]
    else:
        plate_crop = image

    if plate_crop.size == 0:
        return None, 0.0

    # 2. Preprocess the crop
    processed_crop = preprocess_plate(plate_crop)

    # 3. Read the text
    # allowlist restricts the AI to only look for alphanumeric characters
    ocr_results = reader.readtext(processed_crop, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')

    if not ocr_results:
        return None, 0.0

    # 4. Extract the best result (assuming the plate is the most prominent text)
    # readtext returns a list of tuples: (bounding_box, text, confidence)
    best_result = max(ocr_results, key=lambda x: x[2])
    text = best_result[1]
    confidence = best_result[2]

    return text, confidence

In [ ]:
# To test this right now without YOLO, load an image from your dataset
# and either pass it directly (if it's already cropped) or provide a manual dummy box.

def test_ocr_pipeline(image_path):
    # Read the image
    img = cv2.imread(image_path)
    if img is None:
        print(f"❌ Error: Could not load image at {image_path}")
        return

    # NOTE: Since we are testing without YOLO, we are passing bbox=None.
    # This assumes the test image you provide is zoomed in on a car/plate.
    # If it's a massive full-street image, EasyOCR might try to read street signs.
    text, conf = extract_license_plate_text(img, bbox=None)

    # Display the result
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Detected: {text} (Conf: {conf:.2f})" if text else "No text detected")
    plt.axis('off')
    plt.show()

# Replace this with an actual path from your unzipped Kaggle dataset in /content/dataset/
# test_image_path = '/content/dataset/train/images/your_image_name.jpg'
# test_ocr_pipeline(test_image_path)

In [ ]:
def test_plate_reading(image_path):
    # Preprocess the image
    original_img, processed_img = preprocess_for_ocr(image_path)

    # Run EasyOCR on the processed image
    # detail=0 returns just the text. detail=1 returns bounding boxes, text, and confidence scores
    results = reader.readtext(processed_img, detail=1)

    # Plotting the results
    plt.figure(figsize=(10, 5))

    # Show original image
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB))
    plt.title("Original Image")
    plt.axis('off')

    # Show processed image
    plt.subplot(1, 2, 2)
    plt.imshow(processed_img, cmap='gray')
    plt.title("Preprocessed for OCR")
    plt.axis('off')

    plt.show()

    print("\n🔍 OCR Results:")
    if not results:
        print("No text detected.")
    else:
        for (bbox, text, prob) in results:
            print(f"Text: {text} | Confidence: {prob:.2f}")

# --- TEST IT OUT ---
# Replace 'test_image.jpg' with a path to an actual image from your Kaggle dataset
# Example: test_image_path = '/content/dataset/train/images/some_car_image.jpg'

# test_image_path = 'PUT_YOUR_TEST_IMAGE_PATH_HERE.jpg'
# test_plate_reading(test_image_path)